# XGBoost Models

In [1]:
# reading data sets
import pandas as pd
import numpy as np
import os


meta_df = pd.read_csv("data/processed_data/YACHIDA_CRC_2019/metadata.tsv", sep="\t")
meta_df['Study.Group'] = meta_df['Study.Group'].replace({'HS': 'Healthy'})
print(meta_df['Study.Group'].value_counts(dropna=False))

folder_path = r"C:\Users\tonyp\compSci\Research\Multimodal_Microbiome_Fusion_Project\Meta_Fusion\data\processed_data\YACHIDA_CRC_2019\Binary_datasets"

# diseased
MGX_df_diseased = pd.read_csv(
    os.path.join(folder_path, "MGX_df_diseased.tsv"), sep="\t", index_col=0
)
MTB_df_diseased = pd.read_csv(
    os.path.join(folder_path, "MTB_df_diseased.tsv"), sep="\t", index_col=0
)
y_diseased = pd.read_csv(
    os.path.join(folder_path, "y_diseased.tsv"), sep="\t", index_col=0
)["target"]

# MP
MGX_df_MP = pd.read_csv(
    os.path.join(folder_path, "MGX_df_MP.tsv"), sep="\t", index_col=0
)
MTB_df_MP = pd.read_csv(
    os.path.join(folder_path, "MTB_df_MP.tsv"), sep="\t", index_col=0
)
y_MP = pd.read_csv(
    os.path.join(folder_path, "y_MP.tsv"), sep="\t", index_col=0
)["target"]

# Early
MGX_df_early = pd.read_csv(
    os.path.join(folder_path, "MGX_df_early.tsv"), sep="\t", index_col=0
)
MTB_df_early = pd.read_csv(
    os.path.join(folder_path, "MTB_df_early.tsv"), sep="\t", index_col=0
)
y_early = pd.read_csv(
    os.path.join(folder_path, "y_early.tsv"), sep="\t", index_col=0
)["target"]

# Intermediate
MGX_df_intermediate = pd.read_csv(
    os.path.join(folder_path, "MGX_df_intermediate.tsv"), sep="\t", index_col=0
)
MTB_df_intermediate = pd.read_csv(
    os.path.join(folder_path, "MTB_df_intermediate.tsv"), sep="\t", index_col=0
)
y_intermediate = pd.read_csv(
    os.path.join(folder_path, "y_intermediate.tsv"), sep="\t", index_col=0
)["target"]

# Late
MGX_df_late = pd.read_csv(
    os.path.join(folder_path, "MGX_df_late.tsv"), sep="\t", index_col=0
)
MTB_df_late = pd.read_csv(
    os.path.join(folder_path, "MTB_df_late.tsv"), sep="\t", index_col=0
)
y_late = pd.read_csv(
    os.path.join(folder_path, "y_late.tsv"), sep="\t", index_col=0
)["target"]

Study.Group
Healthy         157
Stage_I_II       69
Stage_III_IV     54
MP               40
Stage_0          27
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# 1. Target encoding
y_multiclass = meta_df["Study.Group"]
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_multiclass)

X_mgx = MGX_df_diseased
X_mtb = MTB_df_diseased
num_classes = len(label_encoder.classes_)

# 2. Configure XGBoost
# Let XGBoost use all CPU threads internally (n_jobs=-1), and use tree_method='hist' for massive speedups
xgb_mgx = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    tree_method="hist",  # Histogram-based splitting speeds up high-dimensional data by 5-10x
    n_jobs=-1,  # XGBoost handles parallelism internally
)

xgb_mtb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"auc": "roc_auc_ovr", "accuracy": "accuracy"}

# 3. Run cross_validate with n_jobs=1 (prevents outer thread deadlocks)
print("Evaluating MGX...")
res_mgx = cross_validate(
    xgb_mgx, X_mgx, y_encoded, cv=cv, scoring=scoring, n_jobs=1
)

print("Evaluating MTB...")
res_mtb = cross_validate(
    xgb_mtb, X_mtb, y_encoded, cv=cv, scoring=scoring, n_jobs=1
)

# 4. Print results
print("\n--- Multiclass Results ---")
print(
    f"MGX - Mean AUC: {res_mgx['test_auc'].mean():.4f} (± {res_mgx['test_auc'].std():.4f}) | "
    f"Accuracy: {res_mgx['test_accuracy'].mean():.4f} (± {res_mgx['test_accuracy'].std():.4f})"
)
print(
    f"MTB - Mean AUC: {res_mtb['test_auc'].mean():.4f} (± {res_mtb['test_auc'].std():.4f}) | "
    f"Accuracy: {res_mtb['test_accuracy'].mean():.4f} (± {res_mtb['test_accuracy'].std():.4f})"
)

Evaluating MGX...


KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

y_multiclass = meta_df["Study.Group"]


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_mgx = MGX_df_diseased
X_mtb = MTB_df_diseased

xgb_mgx = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    eval_metric="logloss",
)
xgb_mtb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    eval_metric="logloss",
)


# Compute AUC scores
xgb_mgx_auc = cross_val_score(
    xgb_mgx, X_mgx, y_multiclass, cv=cv, scoring="roc_auc", n_jobs=-1
)
xgb_mtb_auc = cross_val_score(
    xgb_mtb, X_mtb, y_multiclass, cv=cv, scoring="roc_auc", n_jobs=-1
)

# Compute Accuracy scores
mgx_acc_scores = cross_val_score(
    xgb_mgx, X_mgx, y_multiclass, cv=cv, scoring="accuracy", n_jobs=-1
)
mtb_acc_scores = cross_val_score(
    xgb_mtb, X_mtb, y_multiclass, cv=cv, scoring="accuracy", n_jobs=-1
)

# Output results
print("--- Diseased (5-Fold Cross-Validation) ---")
print(
    f"XGBoost - Mean CV AUC (MGX): {xgb_mgx_auc.mean():.4f} (± {xgb_mgx_auc.std():.4f}) | Accuracy: {mgx_acc_scores.mean():.4f} (± {mgx_acc_scores.std():.4f})"
)
print(
    f"XGBoost - Mean CV AUC (MTB): {xgb_mtb_auc.mean():.4f} (± {xgb_mtb_auc.std():.4f}) | Accuracy: {mtb_acc_scores.mean():.4f} (± {mtb_acc_scores.std():.4f})"
)